# Profilage DVF 2022 — Distribution logarithmique de la valeur fonciere (enrichi)

Ce notebook produit l'histogramme de la valeur fonciere en echelle logarithmique,
avec deux lignes de reference : la mediane et la moyenne. L'ecart visuel entre
les deux lignes illustre l'asymetrie de la distribution (ratio 16,1).

Le fichier est enregistre sous un nom different (fig2b) pour ne pas ecraser
l'image existante fig2.

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [7]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "matplotlib", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [8]:
import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.rcParams.update({"font.family": "sans-serif", "font.size": 10})

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Extraction et transformation logarithmique

In [9]:
# Extraction des valeurs foncieres (prix > 0)
vals = con.execute(f"""
    SELECT \"Valeur fonciere\"
    FROM '{pq}'
    WHERE \"Valeur fonciere\" IS NOT NULL
    AND \"Valeur fonciere\" > 0
""").fetchnumpy()["Valeur fonciere"]

# Transformation logarithmique
log_vals = np.log(vals.astype(float) + 1)

# Mediane et moyenne
mediane = int(np.median(vals))
moyenne = int(np.mean(vals))
log_mediane = np.log(mediane + 1)
log_moyenne = np.log(moyenne + 1)
ratio = round(moyenne / mediane, 1)

print(f"Lignes avec prix > 0 : {len(vals):,}".replace(",", " "))
print(f"Mediane : {mediane:>12,} EUR  (ln = {log_mediane:.2f})".replace(",", " "))
print(f"Moyenne : {moyenne:>12,} EUR  (ln = {log_moyenne:.2f})".replace(",", " "))
print(f"Ratio   : {ratio}")

Lignes avec prix > 0 : 4 586 415
Mediane :      175 000 EUR  (ln = 12.07)
Moyenne :    2 825 926 EUR  (ln = 14.85)
Ratio   : 16.1


## Cellule 4 — Graphique avec mediane et moyenne

Deux lignes verticales : rouge pour la mediane, orange pointille pour la moyenne.
L'ecart entre les deux montre visuellement l'asymetrie de la distribution.

In [10]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(log_vals, bins=150, color="#2E75B6", edgecolor="white", linewidth=0.3)

# Mediane (rouge, trait plein)
label_med = f"Mediane ({mediane:,})".replace(",", " ")
ax.axvline(log_mediane, color="red", linewidth=1.5, label=label_med)

# Moyenne (orange, pointille)
label_moy = f"Moyenne ({moyenne:,})".replace(",", " ")
ax.axvline(log_moyenne, color="#E97132", linewidth=1.5, linestyle="--", label=label_moy)

ax.set_xlabel("ln(Valeur fonciere + 1)")
ax.set_ylabel("Nombre de transactions")
ax.set_title("Distribution de la valeur fonciere (echelle logarithmique)")
ax.legend(fontsize=11)

plt.tight_layout()

# Nom different pour ne pas ecraser l'image existante
nom_fichier = "fig2b_distribution_log_VF_enrichi.png"
fig.savefig(SORTIE / nom_fichier, dpi=150, bbox_inches="tight")
print(f"Graphique enregistre : {nom_fichier}")
plt.show()

Graphique enregistre : fig2b_distribution_log_VF_enrichi.png


[chemin temporaire]:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cellule 5 — Reperes pour la lecture du graphique

Correspondance entre les valeurs sur l'axe x (logarithme) et les prix en euros.

In [11]:
reperes = [1, 10, 100, 1000, 5000, 10000, 50000, 100000, 175000, 500000, 1000000, 10000000, 100000000, 1000000000]

col1 = 'Prix (EUR)'
col2 = 'ln(prix + 1)'
print(f"{col1:<20} {col2:>14}")
print("-" * 36)
for prix in reperes:
    ln_val = np.log(prix + 1)
    print(f"  {prix:>15,} EUR {ln_val:>12.2f}".replace(",", " "))

Prix (EUR)             ln(prix + 1)
------------------------------------
                1 EUR         0.69
               10 EUR         2.40
              100 EUR         4.62
            1 000 EUR         6.91
            5 000 EUR         8.52
           10 000 EUR         9.21
           50 000 EUR        10.82
          100 000 EUR        11.51
          175 000 EUR        12.07
          500 000 EUR        13.12
        1 000 000 EUR        13.82
       10 000 000 EUR        16.12
      100 000 000 EUR        18.42
    1 000 000 000 EUR        20.72


## Cellule 6 — Fermeture

In [12]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
